# SCRFD 2.5G + Online SR on Kaggle (2x T4)

This notebook keeps the public `SCRFD-2.5G` architecture fixed and improves only sample redistribution with the online SR scheduler.

Recommended Kaggle settings:
- Internet: ON
- Accelerator: 2x T4 GPU
- Dataset mounted in RetinaFace/SCRFD format


In [ ]:
import os, json, multiprocessing, subprocess, textwrap
import torch

print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))
print('CPU cores:', multiprocessing.cpu_count())


In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/TQC0103/scrfd-fullsearch-kaggle.git /kaggle/working/scrfd-fullsearch-kaggle
%cd /kaggle/working/scrfd-fullsearch-kaggle
!git pull origin main

In [ ]:
# Set this to your mounted Kaggle dataset root in RetinaFace/SCRFD format.
SCRFD_DATA_ROOT = '/kaggle/input/widerface-retinaface-format'

# Throughput-oriented defaults for 2x T4 on Kaggle.
os.environ['SCRFD_DATA_ROOT'] = SCRFD_DATA_ROOT
os.environ['GPU_IDS'] = '0,1'
os.environ['SCRFD_SAMPLES_PER_GPU'] = '8'
os.environ['SCRFD_WORKERS_PER_GPU'] = '4'
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

for key in ['SCRFD_DATA_ROOT', 'GPU_IDS', 'SCRFD_SAMPLES_PER_GPU', 'SCRFD_WORKERS_PER_GPU', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS']:
    print(key, '=', os.environ[key])

In [ ]:
!bash scripts/prepare_env.sh

## Quick sanity run

Runs a short training job to verify the environment, 2-GPU DDP path, and online SR scheduler updates.

In [ ]:
!SCRFD_TOTAL_EPOCHS=16 SCRFD_EVAL_INTERVAL=4 SCRFD_CHECKPOINT_INTERVAL=4 WORK_NAME=scrfd_2.5g_online_sr_2gpu_quick bash scripts/train_scrfd_2.5g_online_sr_2gpu.sh

In [ ]:
!cat /kaggle/working/scrfd-fullsearch-kaggle/work_dirs/scrfd_2.5g_online_sr_2gpu_quick/sr_scheduler_state.json

## Full training

Uncomment and run this when the quick sanity run succeeds.

In [ ]:
# !WORK_NAME=scrfd_2.5g_online_sr_2gpu bash scripts/train_scrfd_2.5g_online_sr_2gpu.sh

In [ ]:
# !cat /kaggle/working/scrfd-fullsearch-kaggle/work_dirs/scrfd_2.5g_online_sr_2gpu/sr_scheduler_state.json